In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PartitionDemo") \
    .getOrCreate()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/17 05:54:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/17 05:54:43 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [2]:
df = spark.range(5000000)

In [4]:
# Initial number of partitions
print("Initial Partitions:", df.rdd.getNumPartitions())

Initial Partitions: 12


In [5]:
# Increase partitions to 12
df_repartitioned = df.repartition(12)
print("Partitions after repartition(12):",
      df_repartitioned.rdd.getNumPartitions())

[Stage 0:>                                                        (0 + 12) / 12]

Partitions after repartition(12): 12


In [6]:
# Reduce partitions to 3
df_coalesced = df_repartitioned.coalesce(3)

print("Partitions after coalesce(3):",
      df_coalesced.rdd.getNumPartitions())

[Stage 1:======================================>                   (8 + 4) / 12]

Partitions after coalesce(3): 3


In [7]:
df_coalesced.show(5)

[Stage 2:=================================>                        (7 + 5) / 12]

+------+
|    id|
+------+
|366363|
|392367|
|281698|
|116978|
|376474|
+------+
only showing top 5 rows



In [8]:
df.write.mode("overwrite").csv("output/data",header = True)

In [9]:
df_repartitioned.write.mode("overwrite").csv("output/data_afterPartitioned",header=True)

In [10]:
df_coalesced.write.mode("overwrite").csv("output/data_coalesced",header=True)

In [11]:
spark.stop()